<a href="https://colab.research.google.com/github/Sebastianelli-Nicola/Traffic-Driven-EV-Queuing-Predictor/blob/main/piano_sperimentale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecasting del Traffico e Occupazione Stazioni di Ricarica EV
**Pipeline Sperimentale e Valutazione dei Modelli Predittivi**

Questo notebook implementa l'architettura di Machine Learning progettata per la previsione a breve termine del traffico veicolare. L'infrastruttura di codice è strutturata per rispondere alle specifiche delineate nel piano di progetto, suddividendo il lavoro in due fasi operative principali.

---

## Fase 2: Definizione del Protocollo Sperimentale

In questa prima fase, l'approccio metodologico ai dati è stato completamente ridefinito per simulare in modo accurato un ambiente di produzione reale, superando i limiti dei test statici.

*   **Implementazione della Sliding Window:** Il tradizionale partizionamento statico dei dati (80% per il training, 20% per il test) è stato abbandonato. Il sistema adotta ora un approccio dinamico basato sulla tecnica della *finestra scorrevole* (Sliding Window).
*   **Impostazione dei Test Indipendenti:** L'infrastruttura isola un periodo di prova finale (es. le ultime due settimane disponibili). All'interno di tale periodo, ogni singola giornata costituisce un test indipendente. Per addestrare il modello in vista di ogni test, il sistema sfrutta quasi tutti i dati storici a disposizione.
*   **Aggregazione Temporale:** L'obiettivo non è una previsione generica, ma puntuale. Per questo motivo, l'ingestione dei dati, l'addestramento e i risultati dei test sono strutturati e aggregati in **slot temporali esatti della durata di 30 minuti**.

---

## Fase 3: Esecuzione Computazionale e Calcolo Metriche

Questa sezione rappresenta il cuore operativo. È progettata per eseguire computazioni intensive in background (elaborazione offline) ed estrarre metriche per un futuro deployment del sistema.

*   **Selezione dei Modelli Predittivi:** Per ora sono stati inseriti solo alcuni modelli (Random Forest, SARIMA, LSTM). A queste è stata integrata la sperimentazione di **Neural Prophet**, un'architettura moderna basata su PyTorch.
*   **Elaborazione Offline Automatizzata:** Le routine di addestramento e validazione (Evaluation Loop) sono incapsulate in script automatizzati. Questo permette l'esecuzione autonoma e non supervisionata di interi batch di calcolo.
*   **Tracciamento delle Tempistiche:** Il sistema registra puntualmente le performance hardware e software. Viene misurato il tempo di *Training* (calcolo pesante eseguito offline) e la latenza di *Predizione*. Quest'ultima metrica è critica: nel sistema a regime, il calcolo predittivo dovrà risultare estremamente veloce per poter essere ricalcolato continuamente in modalità online.
*   **Analisi del Decadimento dell'Errore nel Tempo:** Per determinare il ciclo di vita utile del modello, l'orizzonte di previsione massimo viene spinto fino a **2 o 3 giorni**. Il codice calcola e registra l'errore commesso per *ogni singolo slot di 30 minuti*. Mappando come si modifica tale errore all'allontanarsi dal momento della previsione, è possibile stabilire oggettivamente l'affidabilità temporale del sistema e determinare la frequenza ideale per il riaddestramento.

## 1: Ingestion & Pre-Processing
Questa sezione si occupa di importare i dati grezzi da Google Drive, isolare le misurazioni della singola stazione e aggregare i dati in slot temporali esatti da **30 minuti**. Vengono inoltre estratte le feature temporali di base (ora, giorno, weekend) che i modelli utilizzeranno per apprendere i pattern comportamentali.

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# 1. CONNESSIONE A GOOGLE DRIVE
drive.mount('/content/drive')

# ATTENZIONE: Modifica questo percorso con la tua cartella esatta
PATH = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_ricostruito/'
FILE_NAME = 'traffico_r.csv'

def pre_processing_stazione(nome_file, id_stazione=3):
    print(f"--- BLOCCO 1: Pre-processing Stazione {id_stazione} ---")
    percorso_completo = os.path.join(PATH, nome_file)

    if not os.path.exists(percorso_completo):
        raise FileNotFoundError(f"File {nome_file} non trovato nel percorso {PATH}")

    # Lettura e Setup Indice
    df = pd.read_csv(percorso_completo)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    df.sort_index(inplace=True)

    # Filtraggio
    df_stazione = df[df['id_stazione'] == id_stazione].copy()

    # Aggregazione a 30 Minuti (Resampling)
    df_resampled = df_stazione[['differenza']].resample('30min').mean()
    df_resampled['differenza'] = df_resampled['differenza'].ffill()

    # Generazione Feature Temporali Base
    df_resampled['hour'] = df_resampled.index.hour
    df_resampled['dayofweek'] = df_resampled.index.dayofweek
    df_resampled['is_weekend'] = (df_resampled.index.dayofweek >= 5).astype(int)

    print("Dataset pronto: ", len(df_resampled), "slot da 30 minuti.\n")
    return df_resampled

# Esecuzione
dataset_pulito = pre_processing_stazione(FILE_NAME, id_stazione=3)